# Notebook 4 — Newton: buying iterations with curvature

**Day 4.** Read this after Lecture 4, with `NewtonDirection`, `CholeskySolver` and
`RegularizedObjective` written.

Days 2 and 3 used only the gradient, so the best model of $f$ available was a plane, and
a plane has no minimum — the step length had to come from somewhere else (a fixed
$\alpha$, a line search, an adaptive rule). Newton's method builds a *quadratic* model
instead. A quadratic has a minimum, so the step length comes for free:

$$f(x + d) \;\approx\; f(x) + g^\top d + \tfrac{1}{2} d^\top H d
\qquad\Longrightarrow\qquad
H d = -g .$$

That one change buys an enormous amount and costs three specific things. This notebook
measures all four:

1. **the gain** — 6 iterations where gradient descent needs 568, with the error squaring
   at every step;
2. **cost one** — $H$ must be positive definite, and when it is not, Cholesky *refuses*;
3. **cost two** — forming and factorising $H$ is $O(nd^2 + d^3)$ per step;
4. **cost three** — nothing here fixes a problem whose optimum does not exist; ridge
   does, and the same $\lambda I$ fixes (2) as a side effect.

Every number in a caption is printed by the cell above it.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# The optlab root is the nearest ancestor holding pyproject.toml, so this works whether
# Jupyter was started in notebooks/ or in the repository root.
HERE = Path.cwd()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "pyproject.toml").exists()), HERE.parent)

try:
    import optlab
except ModuleNotFoundError:
    sys.path.insert(0, str(ROOT / "src"))
    import optlab

sys.path.insert(0, str(ROOT))  # for datasets/

np.set_printoptions(precision=6, suppress=True)
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3, "font.size": 10,
                     "axes.titlesize": 10, "figure.dpi": 110})
rng = np.random.default_rng(20250921)

print("optlab root  :", ROOT)
print("optlab loaded:", Path(optlab.__file__).parent)

In [ ]:
def status(name, thunk):
    """Report whether one piece of the package is implemented, without a traceback."""
    try:
        thunk()
    except NotImplementedError:
        return f"  MISSING   {name}"
    except Exception as err:                      # noqa: BLE001 - we want to see anything
        return f"  BROKEN    {name}   ({type(err).__name__}: {err})"
    return f"  ok        {name}"

In [ ]:
from optlab.errors import NotPositiveDefiniteError
from optlab.linalg import CholeskySolver, cholesky
from optlab.linesearch import Armijo
from optlab.objective_ops import RegularizedObjective
from optlab.observers import History
from optlab.optimizers import (DescentOptimizer, NewtonDirection, SteepestDescent,
                               newton)
from optlab.problems import Quadratic, Rosenbrock, logistic_regression
from optlab.regularizers import L2, NoRegularizer
from optlab.stopping import AnyOf, GradientNormBelow, MaxIterations

_A = np.array([[4.0, 1.0], [1.0, 3.0]])
_q = Quadratic(_A, np.array([1.0, -2.0]))
_w = np.zeros(2)

print("Day 4 readiness")
print(status("Quadratic.hessian", lambda: _q.hessian(_w)))
print(status("cholesky", lambda: cholesky(_A)))
print(status("CholeskySolver", lambda: CholeskySolver().solve(_A, np.ones(2))))
print(status("NewtonDirection",
             lambda: NewtonDirection(CholeskySolver()).direction(_q, _w, _q.gradient(_w))))
print(status("newton", lambda: newton(max_iter=3).minimize(_q, _w)))
print(status("NoRegularizer", lambda: NoRegularizer().value(_w)))
print(status("L2", lambda: L2(1.0).value(_w)))
print(status("RegularizedObjective",
             lambda: RegularizedObjective(_q, L2(1.0)).hessian(_w)))
print()
print("Also needed here")
print(status("GLMLoss.hessian",
             lambda: logistic_regression(np.eye(2), np.array([1.0, 0.0])).hessian(_w)))
print(status("Rosenbrock.hessian", lambda: Rosenbrock().hessian(_w)))

## 1. On a quadratic, Newton is not an iterative method

If $f$ is exactly quadratic then the model is exactly $f$, and solving $Hd = -g$ lands on
the minimiser. Not close to it — on it. Your `Quadratic` is
$f(x) = \tfrac{1}{2}x^\top A x - b^\top x$, so the minimiser is $A^{-1}b$, and we can
check the claim against `np.linalg.solve`.

In [ ]:
A = np.array([[4.0, 1.0], [1.0, 3.0]])
b_vec = np.array([1.0, -2.0])
quad = Quadratic(A, b_vec)
x0 = np.array([5.0, -7.0])

res = newton(max_iter=20).minimize(quad, x0)
print(f"started at        {x0}")
print(f"Newton result     {res.x}   after {res.iterations} iteration(s)")
print(f"np.linalg.solve   {np.linalg.solve(A, b_vec)}")
print(f"gradient norm     {res.grad_norm:.3e}")
print(f"converged         {res.converged}")

One iteration, gradient norm $1.8\times10^{-15}$ — the residual of a linear solve, not the
tail of a convergence. Everything after this section is about what happens when the
quadratic model is only an *approximation*.

> **A useful way to hold it.** Gradient descent asks "which way is downhill?". Newton
> asks "where is the bottom of the bowl that best matches $f$ here?" and goes straight
> there. When $f$ *is* a bowl, that is the answer. When $f$ is not, it is a guess — and
> §3 is about how badly that guess can fail.

## 2. Quadratic convergence: the error squares every step

On a smooth function near a minimum with $H \succ 0$, Newton satisfies

$$\|x_{k+1} - x^*\| \;\le\; C\,\|x_k - x^*\|^2 .$$

Squaring the error means *doubling the correct digits* at every iteration. That is a
qualitatively different animal from the constant factor $(\kappa-1)/(\kappa+1)$ of
notebook 2, and it is why the iteration counts below are not a typo.

Same logistic regression for both methods, same Armijo line search, same starting point.

In [ ]:
n, d = 500, 6
data_rng = np.random.default_rng(11)
X = data_rng.normal(size=(n, d))
w_gen = data_rng.normal(size=d) * 1.5
y = (data_rng.random(n) < 1.0 / (1.0 + np.exp(-(X @ w_gen)))).astype(float)

loss = logistic_regression(X, y)
w0 = np.zeros(d)


def trace(direction, problem, start, tol=1e-12, max_iter=50):
    """Run one DescentOptimizer and return its gradient norms, one per iteration."""
    hist = History()
    DescentOptimizer(direction, Armijo(),
                     AnyOf(GradientNormBelow(tol), MaxIterations(max_iter)),
                     observers=[hist]).minimize(problem, np.asarray(start, dtype=float))
    return np.array([e.grad_norm for e in hist.events])


g_newton = trace(NewtonDirection(CholeskySolver()), loss, w0)
g_descent = trace(SteepestDescent(), loss, w0, max_iter=20000)

print(f"Newton stopped after {len(g_newton)} iterations, |g| = {g_newton[-1]:.3e}")
print(f"GD ran {len(g_descent)} iterations, |g| = {g_descent[-1]:.3e}")
print()
print(f"{'target |g|':>12} {'Newton':>10} {'gradient descent':>18}")
for tol in (1e-2, 1e-4, 1e-6, 1e-8, 1e-10):
    hit_n = np.argmax(g_newton <= tol) + 1 if (g_newton <= tol).any() else None
    hit_g = np.argmax(g_descent <= tol) + 1 if (g_descent <= tol).any() else None
    print(f"{tol:12.0e} {str(hit_n):>10} {str(hit_g):>18}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].semilogy(np.arange(1, len(g_newton) + 1), g_newton, "o-", label="Newton")
ax[0].semilogy(np.arange(1, len(g_descent) + 1), g_descent, "-", lw=1,
               label="gradient descent")
ax[0].set_xlim(0, 60)
ax[0].set_xlabel("iteration")
ax[0].set_ylabel(r"$\|\nabla f\|$")
ax[0].set_title("The first 60 iterations")
ax[0].legend(fontsize=8)

logs = np.log10(g_newton)
ax[1].plot(logs[:-1], logs[1:], "o", ms=7, label="Newton steps")
keep = g_newton[1:] > 1e-15                      # drop the float64-limited final step
fit_all = np.polyfit(logs[:-1], logs[1:], 1)
fit_keep = np.polyfit(logs[:-1][keep], logs[1:][keep], 1)
span = np.array([logs[:-1].min(), logs[:-1].max()])
ax[1].plot(span, np.polyval(fit_keep, span), "k-", lw=1,
           label=f"fit, last step dropped: slope {fit_keep[0]:.2f}")
ax[1].plot(span, np.polyval(fit_all, span), "r--", lw=1,
           label=f"fit, all points: slope {fit_all[0]:.2f}")
ax[1].plot(logs[:-1][~keep], logs[1:][~keep], "rx", ms=11,
           label=r"limited by float64, not by Newton")
ax[1].set_xlabel(r"$\log_{10}\|\nabla f(x_k)\|$")
ax[1].set_ylabel(r"$\log_{10}\|\nabla f(x_{k+1})\|$")
ax[1].set_title("Slope 2 would be exact quadratic convergence")
ax[1].legend(fontsize=7, loc="upper left")

fig.suptitle("Figure 1 — Newton vs gradient descent on the same logistic regression")
fig.tight_layout()
plt.show()

**Figure 1, left.** Newton's seven points fall off the bottom of the plot while gradient
descent is still on its first decade. Read the table instead of the picture for the
numbers: to reach $\|g\| \le 10^{-8}$, Newton needs **6** iterations and gradient descent
needs **568**. At $10^{-10}$ gradient descent does not get there at all in its 20 000
iteration budget — it stalls, because by then the decrease per step is below what
float64 can certify and Armijo backtracks to nothing.

**Figure 1, right — and read this one carefully.** Quadratic convergence predicts that
plotting $\log\|g_{k+1}\|$ against $\log\|g_k\|$ gives a slope of 2. The fit over *all*
points gives about $1.55$, which looks like a disappointing result. The fit with the final
step dropped gives $1.91$.

The final step is the one that lands at $\|g\| \approx 3\times10^{-17}$ — the only number
in this section that is not reproducible to the digit, because it is the size of the
rounding error and not the size of anything Newton computed. Newton wanted to go further
than float64 can represent, so that point sits *above* the line the theory predicts, and
including it drags the fitted slope down. The honest reading is that the
theory is confirmed to the limit of the arithmetic and then the arithmetic runs out —
not that the method underperforms.

> This is the general shape of validating a fast method: the last few iterations are
> measuring your floating-point format, not your algorithm. If you fit through them you
> will conclude that every superlinear method is sublinear.

## 3. When Cholesky refuses, and why that is the right behaviour

$Hd = -g$ has a solution whenever $H$ is invertible. But the solution is a *descent*
direction only when $H$ is **positive definite**; if $H$ has a negative eigenvalue, the
quadratic model has a direction that curves downward forever, and its "minimum" is a
saddle or a maximum. Newton will walk towards it.

Cholesky exists precisely to detect this. $H = LL^\top$ requires taking square roots of
the pivots, and on a non-positive-definite matrix a pivot goes negative and there is no
real root. Your `CholeskySolver` raises `NotPositiveDefiniteError` there. The stub
docstring insists you **propagate** it rather than quietly falling back to $-g$, and this
section is why: the exception is information about the problem, not a nuisance.

Rosenbrock has an indefinite region, and it is easy to find analytically. With
$H = \begin{pmatrix} 2 - 400(y - x^2) + 800x^2 & -400x \\ -400x & 200\end{pmatrix}$,
$\det H = 400 - 80000\,y + 80000\,x^2$, so $H$ fails to be positive definite exactly
where $y > x^2 + 1/200$. Let us have your code confirm that.

In [ ]:
ros = Rosenbrock()
grid = 220
xs = np.linspace(-2.0, 2.0, grid)
ys = np.linspace(-1.0, 3.0, grid)
XX, YY = np.meshgrid(xs, ys)

indefinite = np.zeros_like(XX, dtype=bool)
values = np.zeros_like(XX)
for i in range(grid):
    for j in range(grid):
        p = np.array([XX[i, j], YY[i, j]])
        values[i, j] = ros.value(p)
        try:
            cholesky(ros.hessian(p))
        except NotPositiveDefiniteError:
            indefinite[i, j] = True

predicted = YY > XX ** 2 + 1.0 / 200.0
agree = np.mean(indefinite == predicted)
print(f"grid points where Cholesky refuses : {indefinite.sum()} of {indefinite.size}")
print(f"agreement with the region y > x^2 + 1/200 : {agree * 100:.3f}%")
print(f"disagreeing points (all on the boundary)  : {np.sum(indefinite != predicted)}")

In [ ]:
starts = [(-1.2, 1.0), (2.0, -1.0), (0.0, 2.0), (-0.5, 0.5)]
outcomes = {}

for s in starts:
    hist = History()
    try:
        r = DescentOptimizer(NewtonDirection(CholeskySolver()), Armijo(),
                             AnyOf(GradientNormBelow(1e-10), MaxIterations(60)),
                             observers=[hist]).minimize(ros, np.array(s, dtype=float))
        outcomes[s] = ("converged", r.iterations, np.vstack([s] + [e.x for e in hist.events]))
        print(f"from {s}: converged to {np.round(r.x, 6)} in {r.iterations} iterations")
    except NotPositiveDefiniteError:
        path = np.vstack([s] + [e.x for e in hist.events])
        outcomes[s] = ("refused", len(hist.events), path)
        print(f"from {s}: NotPositiveDefiniteError after {len(hist.events)} accepted step(s)")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6))

ax.contour(XX, YY, np.log10(values + 1e-8), levels=25, colors="0.75", linewidths=0.6)
ax.contourf(XX, YY, indefinite.astype(float), levels=[0.5, 1.5],
            colors=["tab:red"], alpha=0.16)
ax.plot(xs, xs ** 2 + 1.0 / 200.0, "r-", lw=1.2,
        label=r"boundary $y = x^2 + 1/200$")

for s, (what, k, path) in outcomes.items():
    colour = "tab:green" if what == "converged" else "tab:red"
    ax.plot(path[:, 0], path[:, 1], "o-", color=colour, ms=3.5, lw=1.1)
    ax.annotate(f"{s}\n{what}", s, textcoords="offset points", xytext=(6, 6),
                fontsize=7.5, color=colour)

ax.plot(1.0, 1.0, "k*", ms=13, label="minimum $(1, 1)$")
ax.set_xlim(-2.15, 2.45)
ax.set_ylim(-1.35, 3.05)
ax.set_xlabel("$x$")
ax.set_ylabel("$y$")
ax.set_title("Figure 2 — Rosenbrock: shaded where the Hessian is not positive definite\n"
             "(green: Newton finished, red: Cholesky refused)")
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()
plt.show()

**Figure 2.** The shaded region is where your `cholesky` raises, and it agrees with
$y > x^2 + 1/200$ on **every one** of the 48 400 grid points. Your code and the algebra
are describing the same set, which is a stronger test of `cholesky` than any unit test in
the suite: it checks the *decision* at 48 400 matrices against a closed form.

Now the four starting points. $(-1.2, 1.0)$ and $(2.0, -1.0)$ are both *outside* the
shaded region and Newton finishes, in 22 and 15 iterations. $(0.0, 2.0)$ and
$(-0.5, 0.5)$ are *inside* it and the very first call raises — after **0** accepted
steps. There is no partial progress to salvage; Newton cannot start.

Follow the green path from $(-1.2, 1.0)$: it runs along the *underside* of the red
boundary for its whole length. The valley of Rosenbrock lies just outside the indefinite
region, so a method that tracks the valley is permanently one bad step away from a
Hessian it cannot factorise. That is the practical reason damping is not optional.

Notice what the classic Rosenbrock starting point $(-1.2, 1.0)$ hides: it sits just below
the parabola, so the textbook demo never meets this failure. Non-convexity is not an
exotic case you can design around — it is most of the plane here, and the standard
example is one of the few points that avoids it.

**What to do about it** is day 4 lab 3: replace $H$ by $H + \tau I$ with $\tau$ large
enough to make it positive definite. As $\tau \to \infty$ the step rotates towards
$-g/\tau$, so the damped method interpolates between Newton and gradient descent. Keep
that sentence in mind for tomorrow — Levenberg–Marquardt is exactly this idea with an
automatic rule for $\tau$.

## 4. What a Newton step costs

Iteration counts are not run times. A gradient descent step on a GLM costs $O(nd)$; a
Newton step additionally forms $X^\top D X$ at $O(nd^2)$ and factorises it at $O(d^3)$.
So Newton is a bargain at $d = 10$ and unusable at $d = 10^6$, and the crossover is
somewhere you should measure rather than guess.

We time the `direction` call itself, which is the honest unit: for `SteepestDescent` that
is a gradient, for `NewtonDirection` it is a gradient, a Hessian and a Cholesky solve.

In [ ]:
import time

dims = [8, 16, 32, 64, 128, 256]
n_fixed = 5000
t_gd, t_newton = [], []

sd = SteepestDescent()
nd = NewtonDirection(CholeskySolver())

print(f"{'d':>5} {'GD step (ms)':>14} {'Newton step (ms)':>18} {'ratio':>8}")
for dim in dims:
    gen = np.random.default_rng(0)
    Xd = gen.normal(size=(n_fixed, dim))
    yd = (gen.random(n_fixed) < 1.0 / (1.0 + np.exp(-(Xd @ gen.normal(size=dim))))).astype(float)
    ld = logistic_regression(Xd, yd)
    wd = np.zeros(dim)

    reps = 30
    t0 = time.perf_counter()
    for _ in range(reps):
        sd.direction(ld, wd, ld.gradient(wd))
    a = (time.perf_counter() - t0) / reps

    t0 = time.perf_counter()
    for _ in range(reps):
        nd.direction(ld, wd, ld.gradient(wd))
    c = (time.perf_counter() - t0) / reps

    t_gd.append(a)
    t_newton.append(c)
    print(f"{dim:5d} {a * 1e3:14.3f} {c * 1e3:18.3f} {c / a:8.2f}")

s_gd = np.polyfit(np.log(dims), np.log(t_gd), 1)[0]
s_nt = np.polyfit(np.log(dims), np.log(t_newton), 1)[0]
print(f"\nmeasured log-log slopes:  GD {s_gd:.2f}   Newton {s_nt:.2f}")
print(f"(n is fixed at {n_fixed}, so the theoretical slopes are 1 and up to 3)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].loglog(dims, np.array(t_gd) * 1e3, "o-", label=f"gradient step, slope {s_gd:.2f}")
ax[0].loglog(dims, np.array(t_newton) * 1e3, "s-", label=f"Newton step, slope {s_nt:.2f}")
ax[0].set_xlabel("dimension $d$")
ax[0].set_ylabel("time per direction (ms)")
ax[0].set_title(f"Cost of one step ($n = {n_fixed}$ fixed)")
ax[0].legend(fontsize=8)

ax[1].semilogx(dims, np.array(t_newton) / np.array(t_gd), "o-")
ax[1].set_xlabel("dimension $d$")
ax[1].set_ylabel("Newton step / GD step")
ax[1].set_title("How many gradient steps a Newton step is worth")
ax[1].axhline(568 / 6, color="k", ls="--", lw=1,
              label="568/6: the §2 iteration ratio")
ax[1].legend(fontsize=8)

fig.suptitle("Figure 3 — iterations are cheap for Newton, steps are not")
fig.tight_layout()
plt.show()

**Figure 3.** These are the only numbers in this notebook that depend on your machine, so
read the shape rather than the digits — on the laptop that wrote this, the ratio climbed
from about $3$ at $d = 8$ to about $42$ at $d = 256$.

The dashed line is $568/6 \approx 95$, the iteration advantage Newton had in §2. As long
as the cost ratio stays *below* that line, Newton wins on wall-clock time, and on this
problem it is still winning comfortably at $d = 256$. Where your curve crosses the line
is where Newton stops paying for itself on your hardware.

Now look at the slopes in the left legend. Against $O(nd)$ and $O(nd^2 + d^3)$ you would
expect $1$ and up to $3$; the measured values are around $0.2$ and $1.0$. That is not an
error in the theory — at these sizes a tuned BLAS is nowhere near its asymptotic regime,
and for the gradient, Python call overhead dominates the arithmetic outright. Complexity
exponents describe a limit, and the limit is much further out than people assume. This is
why "it's $O(d^3)$, don't bother" is a bad reason not to try Newton on a 200-parameter
model, and why the answer is to time it rather than to reason about exponents.

> **The practical rule.** Newton for $d$ in the tens or low hundreds with $n$ moderate —
> which covers most classical statistics, and is why `statsmodels` fits a GLM by IRLS
> (day 4 lab 5) and not by SGD. First-order or quasi-Newton methods beyond that.

## 5. The problem Newton cannot fix: an optimum that does not exist

Here is a failure that has nothing to do with the algorithm. Take logistic regression on
data that a hyperplane separates **perfectly**. For any $w$ that classifies everything
correctly, doubling $w$ makes every predicted probability more confident and strictly
lowers the loss. So $f(2w) < f(w)$, always, and

$$\inf_w f(w) = 0 \text{ is never attained: the maximum-likelihood estimate does not exist.}$$

Watch a *correct* implementation of Newton chase it.

In [ ]:
n_sep, d_sep = 80, 6
sep_rng = np.random.default_rng(5)
X_sep = sep_rng.normal(size=(n_sep, d_sep))
w_sep = np.array([1.5, -1.0, 0.8, 0.0, -0.4, 0.2])
y_sep = (X_sep @ w_sep > 0).astype(float)       # no noise, so separable by construction

separable = logistic_regression(X_sep, y_sep)
w0_sep = np.zeros(d_sep)
print(f"smallest margin in the data: {np.min((2 * y_sep - 1) * (X_sep @ w_sep)):.4f}  "
      "(> 0 means perfectly separable)")

print(f"\n{'budget':>8} {'||w||':>12} {'f(w)':>14} {'||grad||':>12}")
for budget in (5, 10, 20, 40):
    r = DescentOptimizer(NewtonDirection(CholeskySolver()), Armijo(),
                         MaxIterations(budget)).minimize(separable, w0_sep)
    print(f"{budget:8d} {np.linalg.norm(r.x):12.3f} {r.value:14.4e} {r.grad_norm:12.2e}")

Every column is behaving *correctly*: $f$ falls to $10^{-17}$, the gradient norm falls to
$10^{-18}$, and a stopping rule based on either would report success. Meanwhile $\|w\|$
has gone $14.5 \to 59 \to 170 \to 395$ and is still climbing. Give it 200 iterations and
it will keep going until `exp` overflows.

Notice what you do **not** see: no `RuntimeWarning: overflow encountered in exp`, even
though $\|w\|$ has grown until $|Xw|$ is in the hundreds — which is exactly the range that
broke the naive `1/(1 + exp(-z))` in notebook 1 §2. Your guarded `softplus` and `sigmoid`
exponentiate only $-|z|$, so they underflow quietly to zero instead of overflowing. Day
1's stability work is what keeps this cell honest: a warning here would have told you your
arithmetic was failing, and would have hidden the real failure, which is statistical and
silent. The evidence is the $\|w\|$ column, and nothing else was ever going to raise.

This is the failure mode that a gradient-norm stopping criterion cannot see, and it is
common in practice: it happens whenever a feature (or a combination) perfectly predicts
the outcome, which is what you get from a leaked label, a near-duplicated column, or
simply more features than samples.

The fix is not a better optimizer. It is a different objective — add a penalty that
grows with $\|w\|$, so that running away stops being free. Because your
`RegularizedObjective` is an adapter, this costs one line and no change to anything else.

In [ ]:
lams = np.logspace(1, -4, 12)
paths, norms, kappas, iters, converged = [], [], [], [], []

for lam in lams:
    ridge = RegularizedObjective(separable, L2(lam))
    r = newton(tol=1e-10, max_iter=200).minimize(ridge, w0_sep)
    ev = np.linalg.eigvalsh(ridge.hessian(r.x))
    paths.append(r.x)
    norms.append(np.linalg.norm(r.x))
    kappas.append(ev[-1] / ev[0])
    iters.append(r.iterations)
    converged.append(r.converged)

paths = np.array(paths)
print(f"{'lambda':>10} {'||w||':>10} {'kappa(H)':>12} {'iters':>7} {'converged':>10}")
for i, lam in enumerate(lams):
    print(f"{lam:10.2e} {norms[i]:10.4f} {kappas[i]:12.3e} {iters[i]:7d} "
          f"{str(converged[i]):>10}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4))

for j in range(d_sep):
    ax[0].semilogx(lams, paths[:, j], "o-", ms=3,
                   label=f"$w_{j + 1}$ (true {w_sep[j]:+.1f})")
ax[0].axhline(0.0, color="k", lw=0.8)
ax[0].invert_xaxis()
ax[0].set_xlabel(r"$\lambda$ (decreasing $\rightarrow$)")
ax[0].set_ylabel("coefficient")
ax[0].set_title("The ridge path")
ax[0].legend(fontsize=6.5, ncol=2)

ax[1].loglog(lams, norms, "o-")
ax[1].invert_xaxis()
ax[1].set_xlabel(r"$\lambda$ (decreasing $\rightarrow$)")
ax[1].set_ylabel(r"$\|w\|$")
ax[1].set_title(r"$\|w\|$ is finite for every $\lambda > 0$," "\n" r"and diverges as $\lambda \to 0$")

ax[2].loglog(lams, kappas, "o-", label=r"$\kappa(H)$ at the solution")
ax2 = ax[2].twinx()
ax2.semilogx(lams, iters, "s--", color="tab:orange", label="Newton iterations")
ax2.set_ylabel("Newton iterations", color="tab:orange")
ax[2].invert_xaxis()          # twinx shares the axis; inverting both would undo it
ax[2].set_xlabel(r"$\lambda$ (decreasing $\rightarrow$)")
ax[2].set_ylabel(r"$\kappa(H)$")
ax[2].set_title("The same $\\lambda I$ also conditions the Hessian")
ax[2].legend(fontsize=8, loc="upper left")
ax2.legend(fontsize=8, loc="lower right")

fig.suptitle("Figure 4 — ridge on perfectly separable data")
fig.tight_layout()
plt.show()

**Figure 4, left.** Every coefficient shrinks smoothly towards zero as $\lambda$ grows,
and the ordering of the large ones matches the signs that generated the labels. Look at
$w_4$, whose true value is exactly $0.0$: at $\lambda = 10^{-4}$ ridge gives it $0.81$,
not zero. Ridge shrinks but does **not** select; the $\ell_2$ penalty has no kink at the
origin, so nothing pins a coefficient to exactly zero. That is day 6's job, and Figure 1
of notebook 6 is this same plot with $\ell_1$.

**Middle.** $\|w\|$ is $0.035$ at $\lambda = 10$ and $19.75$ at $\lambda = 10^{-4}$ —
finite everywhere, diverging only in the limit $\lambda \to 0$ where we are back to the
unattainable MLE. The penalty has not made the data less separable; it has made running
away cost something.

**Right.** The bonus. $\kappa(H)$ at the solution goes from $1.02$ to $106$ as $\lambda$
falls, and the Newton iteration count tracks it, $2 \to 10$. Adding $\lambda I$ to the
Hessian raises every eigenvalue by $\lambda$, which bounds $\kappa \le (L + \lambda)/\lambda$
*regardless of the data*. So the same one-line change that fixed the statistics also
guarantees Cholesky can never raise `NotPositiveDefiniteError` on this problem again.

That is not a coincidence, and it is the bridge to tomorrow. Levenberg–Marquardt is built
on exactly this observation: if $H + \lambda I$ is always factorisable and $\lambda$
interpolates between Newton and gradient descent, then the only thing left to invent is a
rule for choosing $\lambda$ at each step. Notebook 5 measures that rule.

## Checkpoint

1. Your `CholeskySolver` raises on a Hessian. A colleague proposes catching it and using
   $-g$ instead, "so the optimizer never crashes". Using Figure 2, say what information
   that throws away and what you would do instead.
2. Newton took 6 iterations and gradient descent 568. Using your own Figure 3, at what
   cost ratio does Newton stop being worth it on this problem, and roughly which $d$ is
   that on your machine? Would doubling $n$ move that $d$ up or down?
3. In §5, both $f$ and $\|\nabla f\|$ went to zero while the answer got worse. Write down
   a stopping condition that would have caught it.
4. You add `L2(lam)` and your fit barely changes, whatever `lam` is. Name two things that
   could be true, and one cell from this notebook that distinguishes them.
5. Predict, then check: what does §1 print if you replace `A` with a matrix having one
   negative eigenvalue — for example `np.array([[4.0, 1.0], [1.0, -3.0]])`?

**Before day 5.** Notebook 5 needs `ILeastSquaresProblem` with its `residuals` and
`jacobian`, plus `GaussNewton` and `LevenbergMarquardt`. Note that
`ILeastSquaresProblem` is deliberately **not** an `IObjective` — it exposes residuals and a
Jacobian, not a scalar and a gradient. Lecture 5 explains why that separation is an
interface-segregation decision and not an oversight.